# M21C OL, DA, and DA-minus-OL trend maps

Tile-level exact Theil-Sen trends for June 2000-May 2024. OL and DA use identical paired monthly support. Black stippling marks tiles significant under the autocorrelation-corrected Mann-Kendall test and complete-field BH-FDR at 0.05. Segmented limits use the 98th percentile of absolute slopes; colorbar extensions mark values beyond those limits.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.ticker import ScalarFormatter

import cartopy.crs as ccrs
import cartopy.feature as cfeature

plt.rcParams.update({"font.size": 10, "axes.titlesize": 12, "figure.dpi": 120})

In [ ]:
def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start, *start.parents):
        if (candidate / "projects" / "M21C_ls").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the geosldas-analysis repository root")


REPO_ROOT = find_repo_root()
RESULT_DIR = REPO_ROOT / "projects" / "M21C_ls" / "output" / "trends_breakpoints"
MAP_LAT_MIN = -60.0
ROBUST_PERCENTILE = 98.0
ZERO_HALF_WIDTH_FRACTION_OF_FULL_RANGE = 0.02
LAND_FACE = "0.86"

VARIABLES = {
    "PRECTOTCORRLAND": {"mask": "valid_land", "title": "Precipitation trend", "units": r"kg m$^{-2}$ month$^{-1}$ yr$^{-1}$"},
    "SFMC": {"mask": "valid_land", "title": "Surface soil moisture trend", "units": r"m$^3$ m$^{-3}$ yr$^{-1}$"},
    "RZMC": {"mask": "valid_land", "title": "Root-zone soil moisture trend", "units": r"m$^3$ m$^{-3}$ yr$^{-1}$"},
    "SNOMASLAND": {"mask": "seasonal_snow", "title": "Snow mass trend", "units": r"kg m$^{-2}$ yr$^{-1}$"},
    "SNODPLAND": {"mask": "seasonal_snow", "title": "Snow depth trend", "units": r"m yr$^{-1}$"},
    "FRLANDSNO": {"mask": "seasonal_snow", "title": "Snow-covered fraction trend", "units": r"yr$^{-1}$"},
}

missing = [
    RESULT_DIR / f"{variable}_{series}_{cfg['mask']}_trend_statistics.nc"
    for variable, cfg in VARIABLES.items()
    for series in ("ol", "da", "delta")
    if not (RESULT_DIR / f"{variable}_{series}_{cfg['mask']}_trend_statistics.nc").exists()
]
if missing:
    raise FileNotFoundError("Missing trend products:\n" + "\n".join(map(str, missing)))

print(f"Using trend products from {RESULT_DIR}")

In [ ]:
def centers_to_edges(centers):
    centers = np.asarray(centers, dtype=float)
    middle = 0.5 * (centers[:-1] + centers[1:])
    return np.concatenate(([centers[0] - 0.5 * np.diff(centers)[0]], middle, [centers[-1] + 0.5 * np.diff(centers)[-1]]))


def build_tile_grid(lat, lon, decimals=6):
    lat_round = np.round(np.asarray(lat), decimals)
    lon_round = np.round(np.asarray(lon), decimals)
    lat_unique = np.unique(lat_round)
    lon_unique = np.unique(lon_round)
    i_lat = np.searchsorted(lat_unique, lat_round)
    i_lon = np.searchsorted(lon_unique, lon_round)
    pair_code = i_lat.astype(np.int64) * lon_unique.size + i_lon
    if np.unique(pair_code).size != pair_code.size:
        raise ValueError("Duplicate rounded tile centers prevent grid reconstruction")
    lon_edges, lat_edges = np.meshgrid(centers_to_edges(lon_unique), centers_to_edges(lat_unique))
    return {
        "i_lat": i_lat, "i_lon": i_lon,
        "shape": (lat_unique.size, lon_unique.size),
        "lon_edges": lon_edges, "lat_edges": lat_edges,
    }


def tile_values_to_grid(values, grid):
    output = np.full(grid["shape"], np.nan, dtype=np.float32)
    output[grid["i_lat"], grid["i_lon"]] = np.asarray(values, dtype=np.float32)
    return output


def nice_limit(values, percentile=ROBUST_PERCENTILE):
    values = np.abs(np.asarray(values, dtype=float))
    values = values[np.isfinite(values)]
    raw = float(np.percentile(values, percentile))
    if not np.isfinite(raw) or raw <= 0:
        return 1.0
    power = 10.0 ** np.floor(np.log10(raw))
    return float(np.ceil(raw / power * 5.0) / 5.0 * power)


def segmented_diverging_scale(vmax):
    zero_half_width = 2.0 * vmax * ZERO_HALF_WIDTH_FRACTION_OF_FULL_RANGE
    bounds = np.concatenate((
        np.linspace(-vmax, -zero_half_width, 6),
        [zero_half_width],
        np.linspace(zero_half_width, vmax, 6)[1:],
    ))
    base = plt.get_cmap("RdBu_r")
    colors = [base(value) for value in np.linspace(0.05, 0.40, 5)] + [(1, 1, 1, 1)] + [base(value) for value in np.linspace(0.60, 0.95, 5)]
    cmap = ListedColormap(colors)
    cmap.set_under(colors[0])
    cmap.set_over(colors[-1])
    return cmap, BoundaryNorm(bounds, cmap.N, clip=False), bounds


def load_trend_fields(variable):
    cfg = VARIABLES[variable]
    fields = {}
    reference_lat = reference_lon = None
    for series in ("ol", "da", "delta"):
        path = RESULT_DIR / f"{variable}_{series}_{cfg['mask']}_trend_statistics.nc"
        with xr.open_dataset(path) as dataset:
            lat = np.asarray(dataset["lat"].values)
            lon = np.asarray(dataset["lon"].values)
            if reference_lat is None:
                reference_lat, reference_lon = lat, lon
            elif not (np.array_equal(reference_lat, lat) and np.array_equal(reference_lon, lon)):
                raise ValueError(f"Tile coordinates differ for {variable}/{series}")
            fields[series] = {
                "slope": np.asarray(dataset["slope"].values),
                "significant": np.asarray(dataset["significant_fdr"].values, dtype=bool),
            }
    return reference_lat, reference_lon, fields


def add_map_background(ax):
    ax.set_facecolor("white")
    ax.add_feature(cfeature.LAND.with_scale("110m"), facecolor=LAND_FACE, edgecolor="none", zorder=0)
    ax.add_feature(cfeature.OCEAN.with_scale("110m"), facecolor="white", edgecolor="none", zorder=0)
    ax.coastlines(resolution="110m", linewidth=0.45, color="0.25", zorder=3)
    ax.set_extent([-180, 180, MAP_LAT_MIN, 90], crs=ccrs.PlateCarree())


def add_panel_label(ax, index):
    ax.text(
        0.015, 0.985, f"({chr(97 + index)})", transform=ax.transAxes,
        ha="left", va="top", fontweight="bold", zorder=5,
        bbox=dict(boxstyle="square,pad=0.15", facecolor="white", edgecolor="none", alpha=0.8),
    )


def plot_trend_triptych(variable):
    cfg = VARIABLES[variable]
    lat, lon, fields = load_trend_fields(variable)
    grid = build_tile_grid(lat, lon)
    visible = lat >= MAP_LAT_MIN
    state_values = np.concatenate([fields[name]["slope"][visible] for name in ("ol", "da")])
    delta_values = fields["delta"]["slope"][visible]
    state_vmax = nice_limit(state_values)
    delta_vmax = nice_limit(delta_values)
    state_cmap, state_norm, _ = segmented_diverging_scale(state_vmax)
    delta_cmap, delta_norm, _ = segmented_diverging_scale(delta_vmax)

    fig, axes = plt.subplots(
        1, 3, figsize=(18, 5.4), subplot_kw={"projection": ccrs.Robinson()},
        constrained_layout=True,
    )
    meshes = []
    labels = {"ol": "OL", "da": "DA", "delta": "DA - OL"}
    plate = ccrs.PlateCarree()
    for index, (ax, series) in enumerate(zip(axes, ("ol", "da", "delta"))):
        add_map_background(ax)
        cmap, norm = (state_cmap, state_norm) if series != "delta" else (delta_cmap, delta_norm)
        mesh = ax.pcolormesh(
            grid["lon_edges"], grid["lat_edges"],
            tile_values_to_grid(fields[series]["slope"], grid),
            transform=plate, cmap=cmap, norm=norm, shading="auto",
            rasterized=True, zorder=1,
        )
        meshes.append(mesh)
        significant = fields[series]["significant"] & visible
        ax.scatter(
            lon[significant], lat[significant], s=0.45, c="black", alpha=0.35,
            linewidths=0, transform=plate, rasterized=True, zorder=2,
        )
        ax.set_title(f"{labels[series]}\nFDR significant: {int(significant.sum()):,} tiles")
        add_panel_label(ax, index)

    state_formatter = ScalarFormatter(useMathText=True)
    state_formatter.set_powerlimits((-2, 3))
    delta_formatter = ScalarFormatter(useMathText=True)
    delta_formatter.set_powerlimits((-2, 3))
    state_bar = fig.colorbar(
        meshes[0], ax=axes[:2], orientation="horizontal", extend="both",
        fraction=0.065, pad=0.035, aspect=45, format=state_formatter,
        ticks=np.linspace(-state_vmax, state_vmax, 5),
    )
    state_bar.set_label(f"OL and DA trend ({cfg['units']})")
    delta_bar = fig.colorbar(
        meshes[2], ax=axes[2], orientation="horizontal", extend="both",
        fraction=0.065, pad=0.035, aspect=24, format=delta_formatter,
        ticks=np.linspace(-delta_vmax, delta_vmax, 5),
    )
    delta_bar.set_label(f"DA - OL trend ({cfg['units']})")
    fig.suptitle(cfg["title"], fontsize=15)
    plt.show()
    return fig

## Figure 1: Precipitation trends

Whole-record precipitation trends. Red indicates increasing precipitation and blue decreasing precipitation; white is near zero. Black stippling marks FDR-significant tiles.

In [ ]:
plot_trend_triptych("PRECTOTCORRLAND");

## Figure 2: Surface soil moisture trends

Whole-record surface soil moisture trends. Red indicates wetting and blue drying; white is near zero. Black stippling marks FDR-significant tiles.

In [ ]:
plot_trend_triptych("SFMC");

## Figure 3: Root-zone soil moisture trends

Whole-record root-zone soil moisture trends. Red indicates wetting and blue drying; white is near zero. Black stippling marks FDR-significant tiles.

In [ ]:
plot_trend_triptych("RZMC");

## Figure 4: Snow mass trends

Whole-record snow mass trends over the seasonal-snow domain. Red indicates increasing snow mass and blue decreasing snow mass; white is near zero. Black stippling marks FDR-significant tiles.

In [ ]:
plot_trend_triptych("SNOMASLAND");

## Figure 5: Snow depth trends

Whole-record snow depth trends over the seasonal-snow domain. Red indicates increasing snow depth and blue decreasing snow depth; white is near zero. Black stippling marks FDR-significant tiles.

In [ ]:
plot_trend_triptych("SNODPLAND");

## Figure 6: Snow-covered fraction trends

Whole-record snow-covered fraction trends over the seasonal-snow domain. Red indicates increasing coverage and blue decreasing coverage; white is near zero. Black stippling marks FDR-significant tiles.

In [ ]:
plot_trend_triptych("FRLANDSNO");